# Crawl Data — VnExpress + Tuổi Trẻ
Mục tiêu: **10,000–15,000 bài** chất lượng cao từ 2 nguồn báo lớn.

**Yêu cầu Kaggle:** Settings > Internet > **ON**

Mỗi bài phải có đủ 3 thứ mới được giữ: `content` (≥300 từ) + `summary` (≥40 ký tự) + `tags` (≥2 tag)

In [ ]:
!pip install requests beautifulsoup4 tqdm -q

In [ ]:
import requests
from bs4 import BeautifulSoup
import json, time, re, random
from tqdm import tqdm
from pathlib import Path
from collections import Counter

OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(exist_ok=True)

# Session dùng chung — giữ cookies giữa các request
SESSION = requests.Session()
SESSION.headers.update({
    'User-Agent'     : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
    'Accept'         : 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
    'Accept-Language': 'vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection'     : 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
    'Sec-Fetch-Dest' : 'document',
    'Sec-Fetch-Mode' : 'navigate',
    'Sec-Fetch-Site' : 'none',
    'Cache-Control'  : 'max-age=0',
})

# Lấy cookies bằng cách visit trang chủ trước
print('Khởi tạo session (visit homepage)...')
try:
    SESSION.get('https://vnexpress.net', timeout=10)
    print('  VnExpress ✓')
    time.sleep(1)
    SESSION.get('https://tuoitre.vn', timeout=10)
    print('  Tuổi Trẻ ✓')
except Exception as e:
    print(f'  Warning: {e}')

print('Session sẵn sàng.')

In [ ]:
NOISE_PATTERNS = [
    r'Mời bạn .{0,60}bình luận',
    r'Theo dõi .{0,60}tại đây',
    r'\(Nguồn:.{0,80}\)',
    r'Video:.{0,100}',
    r'Xem thêm:.{0,100}',
    r'Đọc thêm:.{0,100}',
    r'(?i)advertisement',
]

def clean_text(text: str) -> str:
    for pat in NOISE_PATTERNS:
        text = re.sub(pat, '', text)
    return re.sub(r'\s+', ' ', text).strip()

def is_valid(article: dict) -> bool:
    word_count = len(article['content'].split())
    return (
        word_count >= 150          # giảm từ 300 → 150
        and len(article['summary']) >= 20  # giảm từ 40 → 20
        # tags không bắt buộc nữa
    )

def get_text_first(soup, selectors):
    """Thử lần lượt nhiều selector, trả về text của cái tìm thấy đầu tiên"""
    for tag, attr, val in selectors:
        if attr == 'class':
            el = soup.find(tag, class_=val)
        elif attr is None:
            el = soup.find(tag)
        else:
            el = soup.find(tag, attrs={attr: val})
        if el:
            return el
    return None

def get_all_text(soup, selectors):
    """Thử nhiều selector, lấy tất cả <p> bên trong"""
    for tag, attr, val in selectors:
        if attr == 'class':
            el = soup.find(tag, class_=val)
        elif attr is None:
            el = soup.find(tag)
        else:
            el = soup.find(tag, attrs={attr: val})
        if el:
            return el
    return None

In [ ]:
VNE_CATEGORIES = [
    'https://vnexpress.net/khoa-hoc',
    'https://vnexpress.net/so-hoa',
    'https://vnexpress.net/kinh-doanh',
    'https://vnexpress.net/giao-duc',
    'https://vnexpress.net/suc-khoe',
    'https://vnexpress.net/the-gioi',
    'https://vnexpress.net/giai-tri',
    'https://vnexpress.net/phap-luat',
    'https://vnexpress.net/du-lich',
    'https://vnexpress.net/xe',
    'https://vnexpress.net/the-thao',
    'https://vnexpress.net/moi-truong',
    'https://vnexpress.net/tam-su',
    'https://vnexpress.net/gia-dinh',
    'https://vnexpress.net/bat-dong-san',
    'https://vnexpress.net/cong-dong',
]

def vne_get_links(category_url, num_pages=35):
    links = []
    for page in range(1, num_pages + 1):
        url = category_url if page == 1 else f"{category_url}-p{page}"
        try:
            r = SESSION.get(url, timeout=15,
                            headers={'Referer': 'https://vnexpress.net/'})
            if r.status_code != 200:
                break
            soup = BeautifulSoup(r.text, 'html.parser')
            found = False
            for tag in ['h3', 'h2', 'h1']:
                items = soup.find_all(tag, class_='title-news')
                if items:
                    for it in items:
                        a = it.find('a')
                        if a and a.get('href', '').startswith('https://vnexpress.net'):
                            href = a['href'].split('?')[0]   # bỏ query string
                            if '/video/' not in href and '/photo/' not in href and href not in links:
                                links.append(href)
                    found = True
                    break
            if not found:
                break
            time.sleep(random.uniform(0.5, 1.0))
        except Exception:
            break
    return links

def vne_crawl(url):
    try:
        r = SESSION.get(url, timeout=15,
                        headers={'Referer': 'https://vnexpress.net/'})
        if r.status_code != 200:
            return None
        soup = BeautifulSoup(r.text, 'html.parser')

        # ── Title: thử nhiều selector ─────────────────────────────────────
        title_el = get_text_first(soup, [
            ('h1', 'class', 'title-detail'),
            ('h1', 'class', 'title_detail'),
            ('h1', 'itemprop', 'headline'),
            ('h1', None, None),
        ])
        if not title_el: return None
        title = title_el.get_text(strip=True)

        # ── Summary ───────────────────────────────────────────────────────
        sum_el = get_text_first(soup, [
            ('p', 'class', 'description'),
            ('p', 'class', 'lead'),
            ('div', 'class', 'description'),
            ('p', 'itemprop', 'description'),
        ])
        if sum_el:
            summary = clean_text(sum_el.get_text(strip=True))
        else:
            summary = ''

        # ── Tags (không bắt buộc) ─────────────────────────────────────────
        tags = []
        for sel_tag, sel_cls in [('ul','tags-news'), ('div','tags-news'),
                                  ('ul','list-tag'),  ('div','tags'),
                                  ('section','tag'),  ('div','tag')]:
            c = soup.find(sel_tag, class_=sel_cls)
            if c:
                tags = [a.get_text(strip=True) for a in c.find_all('a') if a.get_text(strip=True)]
                if tags: break
        # fallback: lấy meta keywords
        if not tags:
            meta = soup.find('meta', attrs={'name': 'keywords'})
            if meta and meta.get('content'):
                tags = [t.strip() for t in meta['content'].split(',') if t.strip()][:6]

        # ── Content ───────────────────────────────────────────────────────
        body = get_all_text(soup, [
            ('article', 'class', 'fck_detail'),
            ('div',     'class', 'fck_detail'),
            ('div',     'class', 'article-body'),
            ('div',     'itemprop', 'articleBody'),
            ('section', 'class', 'article__body'),
        ])
        if not body:
            # fallback: lấy toàn bộ <p> trong main
            main = soup.find('main') or soup.find('div', id='main_detail')
            if not main: return None
            body = main

        # Xóa element không cần thiết
        for el in body.find_all(['script', 'style', 'aside', 'figure']):
            el.decompose()

        paras = body.find_all('p')
        content = clean_text(' '.join(p.get_text(strip=True) for p in paras if len(p.get_text(strip=True)) > 20))

        # Nếu không có summary thì lấy câu đầu content
        if not summary and content:
            summary = content.split('.')[0].strip()[:300]

        art = {
            'title'  : title,
            'content': content,
            'summary': summary,
            'tags'   : [tg for tg in tags[:6] if len(tg) > 1],
            'source' : 'vnexpress',
        }
        return art if is_valid(art) else None
    except Exception:
        return None

In [ ]:
# ══════════════════════════════════════════════════════════════
# CRAWLER 2: TUỔI TRẺ
# ══════════════════════════════════════════════════════════════

TTO_CATEGORIES = [
    'https://tuoitre.vn/khoa-hoc.htm',
    'https://tuoitre.vn/cong-nghe.htm',
    'https://tuoitre.vn/giao-duc.htm',
    'https://tuoitre.vn/suc-khoe.htm',
    'https://tuoitre.vn/the-gioi.htm',
    'https://tuoitre.vn/van-hoa.htm',
    'https://tuoitre.vn/the-thao.htm',
    'https://tuoitre.vn/phap-luat.htm',
    'https://tuoitre.vn/kinh-doanh.htm',
    'https://tuoitre.vn/du-lich.htm',
    'https://tuoitre.vn/moi-truong.htm',
    'https://tuoitre.vn/nhip-song-tre.htm',
]

def tto_get_links(category_url, num_pages=25):
    links = []
    base = category_url.replace('.htm', '')
    for page in range(1, num_pages + 1):
        url = category_url if page == 1 else f"{base}/trang-{page}.htm"
        try:
            r = requests.get(url, headers=HEADERS, timeout=15)
            if r.status_code != 200:
                break
            soup = BeautifulSoup(r.text, 'html.parser')
            # Tuổi Trẻ article links trong h3 hoặc h2 có class chứa 'title'
            for a in soup.find_all('a', href=True):
                href = a['href']
                if not href.startswith('http'):
                    href = 'https://tuoitre.vn' + href
                if (
                    'tuoitre.vn' in href
                    and href.endswith('.htm')
                    and href not in links
                    and '/trang-' not in href
                    and sum(c == '/' for c in href.split('tuoitre.vn')[-1]) >= 2  # dạng /chuyen-muc/bai-viet.htm
                ):
                    links.append(href)
            time.sleep(random.uniform(0.8, 1.5))
        except Exception:
            break
    return list(set(links))

def tto_crawl(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        if r.status_code != 200:
            return None
        soup = BeautifulSoup(r.text, 'html.parser')

        # Title — thử nhiều selector
        t = (soup.find('h1', class_='article-title')
             or soup.find('h1', attrs={'data-role': 'article-title'})
             or soup.find('h1', class_='title'))
        if not t: return None
        title = t.get_text(strip=True)

        # Summary / sapo
        s = (soup.find('div', class_='sapo')
             or soup.find('p', class_='sapo')
             or soup.find('h2', class_='sapo')
             or soup.find('div', attrs={'data-role': 'sapo'}))
        if not s: return None
        summary = clean_text(s.get_text(strip=True))

        # Tags
        tags = []
        for sel in [('div','tag-keywords'), ('ul','list-tag'), ('div','article-tag'), ('div','tags')]:
            c = soup.find(sel[0], class_=sel[1])
            if c:
                tags = [a.get_text(strip=True) for a in c.find_all('a') if a.get_text(strip=True)]
                if tags: break

        # Content
        body = (soup.find('div', class_='detail-content')
                or soup.find('div', attrs={'data-role': 'content'})
                or soup.find('div', class_='content-detail'))
        if not body: return None

        # Xóa các div quảng cáo / liên quan trước khi lấy text
        for unwanted in body.find_all(['script', 'figure', 'aside', 'div'], class_=re.compile(r'ads|relate|banner|social')):
            unwanted.decompose()

        paras = body.find_all('p')
        content = clean_text(' '.join(p.get_text(strip=True) for p in paras if p.get_text(strip=True)))

        art = {'title': title, 'content': content, 'summary': summary,
               'tags': [tg for tg in tags[:6] if len(tg) > 1], 'source': 'tuoitre'}
        return art if is_valid(art) else None
    except Exception:
        return None

In [ ]:
# ══════════════════════════════════════════════════════════════
# DEBUG: Test crawl 3 bài trước khi chạy toàn bộ
# Chạy cell này để kiểm tra crawler có hoạt động không
# ══════════════════════════════════════════════════════════════
TEST_URLS = [
    'https://vnexpress.net/nhung-cuoc-goi-luc-nua-dem-4862803.html',
    'https://vnexpress.net/khoa-hoc',  # trang danh mục
]

print('=== DEBUG: Test VnExpress crawler ===\n')
for test_url in TEST_URLS[:1]:
    print(f'URL: {test_url}')
    r = SESSION.get(test_url, timeout=15)
    print(f'Status: {r.status_code}')
    soup = BeautifulSoup(r.text, 'html.parser')

    # Kiểm tra từng thành phần
    h1 = soup.find('h1')
    print(f'h1 đầu tiên: {h1.get_text(strip=True)[:80] if h1 else "KHÔNG TÌM THẤY"}')

    title_cls = soup.find('h1', class_='title-detail')
    print(f'h1.title-detail: {"✓ " + title_cls.get_text(strip=True)[:60] if title_cls else "✗ không có"}')

    desc = soup.find('p', class_='description')
    print(f'p.description  : {"✓ " + desc.get_text(strip=True)[:60] if desc else "✗ không có"}')

    fck = soup.find('article', class_='fck_detail') or soup.find('div', class_='fck_detail')
    print(f'fck_detail     : {"✓ tìm thấy" if fck else "✗ không có"}')

    tags_el = soup.find('ul', class_='tags-news') or soup.find('div', class_='tags-news')
    print(f'tags-news      : {"✓ tìm thấy" if tags_el else "✗ không có"}')

    meta_kw = soup.find('meta', attrs={'name': 'keywords'})
    print(f'meta keywords  : {"✓ " + meta_kw["content"][:60] if meta_kw else "✗ không có"}')

    # Thử crawl thực sự
    print('\n--- Thử vne_crawl() ---')
    result = vne_crawl(test_url)
    if result:
        print(f'✓ OK!')
        print(f'  Title  : {result["title"][:80]}')
        print(f'  Summary: {result["summary"][:100]}')
        print(f'  Tags   : {result["tags"]}')
        print(f'  Words  : {len(result["content"].split())}')
    else:
        print('✗ Trả về None — cần kiểm tra thêm')
        # In HTML thô để debug
        print('\nHTML classes của các h1:')
        for el in soup.find_all('h1')[:3]:
            print(f'  {el.get("class")} → {el.get_text(strip=True)[:50]}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# BƯỚC 1: Thu thập links từ cả 2 nguồn
# ══════════════════════════════════════════════════════════════
print('=' * 58)
print('BƯỚC 1: Thu thập links bài viết')
print('=' * 58)

all_links = []  # list of (url, crawler_fn)

print('\n[VnExpress] 16 chuyên mục × 35 trang...')
for cat in VNE_CATEGORIES:
    name = cat.split('/')[-1]
    links = vne_get_links(cat, num_pages=35)
    all_links.extend((l, 'vne') for l in links)
    print(f'  {name:20s} → {len(links)} links')
    time.sleep(random.uniform(1, 2))

print(f'\n[Tuổi Trẻ] 12 chuyên mục × 25 trang...')
for cat in TTO_CATEGORIES:
    name = cat.split('/')[-1].replace('.htm', '')
    links = tto_get_links(cat, num_pages=25)
    all_links.extend((l, 'tto') for l in links)
    print(f'  {name:20s} → {len(links)} links')
    time.sleep(random.uniform(1, 2))

# Bỏ trùng URL
seen = set()
unique_links = []
for item in all_links:
    if item[0] not in seen:
        seen.add(item[0])
        unique_links.append(item)

random.shuffle(unique_links)
print(f'\nTổng links (sau lọc trùng): {len(unique_links)}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# BƯỚC 2: Crawl nội dung bài viết
# ══════════════════════════════════════════════════════════════
print('=' * 58)
print('BƯỚC 2: Crawl nội dung bài viết')
print('=' * 58)

CRAWLERS = {'vne': vne_crawl, 'tto': tto_crawl}

articles = []
stats = {'vne': {'ok': 0, 'fail': 0}, 'tto': {'ok': 0, 'fail': 0}}

for i, (url, source) in enumerate(tqdm(unique_links, desc='Crawling')):
    art = CRAWLERS[source](url)
    if art:
        articles.append(art)
        stats[source]['ok'] += 1
    else:
        stats[source]['fail'] += 1

    time.sleep(random.uniform(0.3, 0.8))

    # Checkpoint mỗi 500 bài
    if (i + 1) % 500 == 0:
        with open(OUTPUT_DIR / 'checkpoint.json', 'w', encoding='utf-8') as f:
            json.dump(articles, f, ensure_ascii=False)
        tqdm.write(f'  [{i+1}/{len(unique_links)}] ok={len(articles)} | vne={stats["vne"]["ok"]} tto={stats["tto"]["ok"]}')

print(f'\nKết quả:')
print(f'  VnExpress : {stats["vne"]["ok"]} bài hợp lệ / {stats["vne"]["fail"]} bị loại')
print(f'  Tuổi Trẻ  : {stats["tto"]["ok"]} bài hợp lệ / {stats["tto"]["fail"]} bị loại')
print(f'  TỔNG      : {len(articles)} bài')

In [ ]:
# ══════════════════════════════════════════════════════════════
# BƯỚC 3: Lọc trùng theo title + Lưu dataset
# ══════════════════════════════════════════════════════════════

# Lọc bài trùng tiêu đề (cùng bài đăng 2 nguồn)
seen_titles = set()
deduped = []
for art in articles:
    key = re.sub(r'\s+', '', art['title'].lower())[:60]
    if key not in seen_titles:
        seen_titles.add(key)
        deduped.append(art)

print(f'Sau dedup: {len(deduped)} bài (loại {len(articles)-len(deduped)} trùng)')

# Thống kê
from collections import Counter
src_count = Counter(a['source'] for a in deduped)
word_counts = [len(a['content'].split()) for a in deduped]
tag_counts  = [len(a['tags']) for a in deduped]

print(f'\nNguồn : VnExpress={src_count["vnexpress"]} | Tuổi Trẻ={src_count["tuoitre"]}')
print(f'Content: min={min(word_counts)} | avg={sum(word_counts)//len(word_counts)} | max={max(word_counts)} từ')
print(f'Tags   : avg={sum(tag_counts)/len(tag_counts):.1f} tag/bài')

# Lưu
out = OUTPUT_DIR / 'vnexpress_dataset.json'
with open(out, 'w', encoding='utf-8') as f:
    json.dump(deduped, f, ensure_ascii=False, indent=2)
print(f'\nDataset saved: {out}')

# Xem mẫu
for i, s in enumerate(deduped[:2]):
    print(f'\n--- Mẫu {i+1} [{s["source"]}] ---')
    print(f'Title  : {s["title"]}')
    print(f'Summary: {s["summary"][:120]}...')
    print(f'Tags   : {s["tags"]}')
    print(f'Content: {" ".join(s["content"].split()[:30])}...')